In [2]:
import pandas as pd
from pathlib import Path
import numpy as np

In [3]:
# Project directories
project_dir = Path("E:/Cloud_cost_capstone")

data_dir = project_dir / "data/raw/fastStorage/2013-8"

selection_file = project_dir / "reports/selected_vm_ids.csv"

# Verify the paths
print("Dataset folder exists:", data_dir.exists())
print("Selection file exists:", selection_file.exists())

Dataset folder exists: True
Selection file exists: True


In [4]:
selected_df = pd.read_csv(selection_file)

print("Shape", selected_df.shape)
print("Coloumns", selected_df.columns.tolist())

display(selected_df.head())

Shape (100, 2)
Coloumns ['VM_ID', 'Filename']


,VM_ID,Filename
0,3,3.csv
1,17,17.csv
2,23,23.csv
3,28,28.csv
4,36,36.csv


In [5]:
# Read the saved identifiers from the first column
selected_ids = selected_df.iloc[:, 0].astype(str).str.strip()

# Find all available VM CSV files
all_files = list(data_dir.rglob("*.csv"))

# Create a lookup: VM ID -> file path
file_lookup = {file.stem: file for file in all_files}

# Normalize identifiers and retrieve files
vm_ids = [
    str(int(float(value))) if value.replace(".", "", 1).isdigit()
    else Path(value).stem
    for value in selected_ids
]

selected_files = [file_lookup[vm_id] for vm_id in vm_ids]

print("Selected VM files:", len(selected_files))
print("Unique VM files:", len(set(selected_files)))
print("First 5 files:", [f.name for f in selected_files[:5]])

Selected VM files: 100
Unique VM files: 100
First 5 files: ['3.csv', '17.csv', '23.csv', '28.csv', '36.csv']


In [6]:
validation_results = []

for file in selected_files:

    # Load one VM at a time
    vm = pd.read_csv(file, sep=";")

    # Remove unnecessary whitespace in column names
    vm.columns = vm.columns.str.strip()

    # Convert timestamps to UTC datetime
    timestamps = pd.to_datetime(
        vm["Timestamp [ms]"],
        unit="s",
        utc=True,
        errors="coerce"
    )

    # Calculate consecutive time differences
    time_diffs = timestamps.sort_values().diff().dt.total_seconds()

    # Count intervals outside 299–301 seconds
    irregular_intervals = (
        time_diffs.notna()
        & ~time_diffs.between(299, 301)
    ).sum()

    # Calculate zero-memory percentage
    zero_memory_pct = (
        vm["Memory usage [KB]"] == 0
    ).mean() * 100

    validation_results.append({
        "vm_id": file.stem,
        "total_rows": len(vm),
        "missing_cells": int(vm.isna().sum().sum()),
        "duplicate_timestamps": int(
            vm["Timestamp [ms]"].duplicated().sum()
        ),
        "invalid_timestamps": int(timestamps.isna().sum()),
        "irregular_intervals": int(irregular_intervals),
        "zero_memory_pct": round(zero_memory_pct, 2),
        "start_time": timestamps.min(),
        "end_time": timestamps.max()
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df.head(10))

,vm_id,total_rows,missing_cells,duplicate_timestamps,invalid_timestamps,irregular_intervals,zero_memory_pct,start_time,end_time
0,3,8634,0,0,0,5,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
1,17,8636,0,0,0,5,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
2,23,8614,0,0,0,20,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
3,28,8634,0,0,0,5,8.17,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
4,36,8620,0,0,0,14,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
5,40,8616,0,0,0,14,0.07,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
6,55,8632,0,0,0,9,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
7,59,41,0,0,0,0,39.02,2013-08-28 10:28:04+00:00,2013-08-28 13:48:05+00:00
8,60,8635,0,0,0,6,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
9,63,8619,0,0,0,16,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00


In [7]:
print("VMs checked:", len(validation_df))

print("Total observations:", validation_df["total_rows"].sum())

print("VMs with missing cells:",
      (validation_df["missing_cells"] > 0).sum())

print("VMs with duplicate timestamps:",
      (validation_df["duplicate_timestamps"] > 0).sum())

print("VMs with irregular intervals:",
      (validation_df["irregular_intervals"] > 0).sum())

print("VMs with more than 50% zero-memory readings:",
      (validation_df["zero_memory_pct"] > 50).sum())

display(validation_df.describe())

VMs checked: 100
Total observations: 890306
VMs with missing cells: 0
VMs with duplicate timestamps: 8
VMs with irregular intervals: 98
VMs with more than 50% zero-memory readings: 11


,total_rows,missing_cells,duplicate_timestamps,invalid_timestamps,irregular_intervals,zero_memory_pct
count,100.000000,100.0,100.000000,100.0,100.000000,100.000000
mean,8903.060000,0.0,323.290000,0.0,1022.210000,13.111800
std,3049.932097,0.0,1270.082255,0.0,3792.604647,25.164191
min,41.000000,0.0,0.000000,0.0,0.000000,0.000000
25%,8614.000000,0.0,0.000000,0.0,6.000000,0.000000
50%,8617.000000,0.0,0.000000,0.0,16.000000,0.065000
75%,8635.000000,0.0,0.000000,0.0,18.000000,10.087500
max,23687.000000,0.0,7547.000000,0.0,22501.000000,98.250000


In [8]:
# Identify VMs with potential data-quality concerns
problem_vms = validation_df[
    (validation_df["missing_cells"] > 0) |
    (validation_df["duplicate_timestamps"] > 0) |
    (validation_df["invalid_timestamps"] > 0) |
    (validation_df["irregular_intervals"] > 0) |
    (validation_df["zero_memory_pct"] > 50)
].copy()

print("VMs requiring investigation:", len(problem_vms))

display(problem_vms.head(10))

VMs requiring investigation: 99


,vm_id,total_rows,missing_cells,duplicate_timestamps,invalid_timestamps,irregular_intervals,zero_memory_pct,start_time,end_time
0,3,8634,0,0,0,5,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
1,17,8636,0,0,0,5,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
2,23,8614,0,0,0,20,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
3,28,8634,0,0,0,5,8.17,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
4,36,8620,0,0,0,14,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
5,40,8616,0,0,0,14,0.07,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
6,55,8632,0,0,0,9,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
8,60,8635,0,0,0,6,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
9,63,8619,0,0,0,16,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00
10,64,11261,0,27,0,5202,0.00,2013-08-12 13:40:46+00:00,2013-09-11 13:39:58+00:00


In [10]:
duplicate_vms = validation_df[
    validation_df["duplicate_timestamps"] > 0
].sort_values("duplicate_timestamps", ascending=False)

display(
    duplicate_vms[
        ["vm_id", "total_rows",
         "duplicate_timestamps", "irregular_intervals"]
    ]
)

,vm_id,total_rows,duplicate_timestamps,irregular_intervals
12,73,23687,7547,22501
52,670,20284,5859,17391
49,625,17370,4404,13044
50,643,17384,4404,13065
51,652,17368,4404,13036
47,614,17270,4346,12893
16,197,11286,1338,3958
10,64,11261,27,5202


In [11]:
# Select the VM with the most duplicate timestamps
vm_id = duplicate_vms.iloc[0]["vm_id"]

file = file_lookup[vm_id]

vm = pd.read_csv(file, sep=";")
vm.columns = vm.columns.str.strip()

# Identify records sharing the same timestamp
duplicate_rows = vm[
    vm.duplicated(subset=["Timestamp [ms]"], keep=False)
].sort_values("Timestamp [ms]")

print("VM being inspected:", vm_id)
print("Total rows:", len(vm))
print("Exact duplicate rows:", vm.duplicated().sum())
print("Repeated timestamp occurrences:",
      vm["Timestamp [ms]"].duplicated().sum())

display(duplicate_rows.head(10))

VM being inspected: 73
Total rows: 23687
Exact duplicate rows: 7546
Repeated timestamp occurrences: 7547


,Timestamp [ms],CPU cores,CPU capacity provisioned [MHZ],CPU usage [MHZ],CPU usage [%],Memory capacity provisioned [KB],Memory usage [KB],Disk read throughput [KB/s],Disk write throughput [KB/s],Network received throughput [KB/s],Network transmitted throughput [KB/s]
1105,1376646375,1,2599.999309,25.999993,1.0,2097152.0,167770.0,0.066667,7.133333,0.0,0.5
1106,1376646375,0,0.000000,0.000000,0.0,2097152.0,167770.0,0.000000,0.000000,0.0,0.5
1107,1376646675,0,0.000000,0.000000,0.0,0.0,167770.0,0.000000,0.000000,0.0,0.0
1108,1376646675,0,0.000000,0.000000,0.0,0.0,167770.0,0.000000,0.000000,0.0,0.0
1109,1376646675,0,0.000000,0.000000,0.0,0.0,167770.0,0.000000,0.000000,0.0,0.0
1110,1376646975,0,0.000000,0.000000,0.0,0.0,167770.0,0.000000,0.000000,0.0,0.0
1111,1376646975,0,0.000000,0.000000,0.0,0.0,167770.0,0.000000,0.000000,0.0,0.0
1112,1376646975,0,0.000000,0.000000,0.0,0.0,167770.0,0.000000,0.000000,0.0,0.0
1113,1376647275,0,0.000000,0.000000,0.0,0.0,167770.0,0.000000,0.000000,0.0,0.0
1114,1376647275,0,0.000000,0.000000,0.0,0.0,167770.0,0.000000,0.000000,0.0,0.0


In [12]:
interval_results = []

for file in selected_files:

    vm = pd.read_csv(file, sep=";")
    vm.columns = vm.columns.str.strip()

    timestamps = pd.to_datetime(
        vm["Timestamp [ms]"],
        unit="s",
        utc=True,
        errors="coerce"
    )

    time_diffs = (
        timestamps.dropna()
        .sort_values()
        .diff()
        .dt.total_seconds()
        .dropna()
    )

    interval_results.append({
        "vm_id": file.stem,
        "duplicate_intervals": int((time_diffs == 0).sum()),
        "normal_intervals": int(
            time_diffs.between(299, 301).sum()
        ),
        "short_intervals": int(
            ((time_diffs > 0) & (time_diffs < 299)).sum()
        ),
        "long_intervals": int(
            ((time_diffs > 301) & (time_diffs < 600)).sum()
        ),
        "large_gaps": int((time_diffs >= 600).sum()),
        "max_gap_seconds": time_diffs.max()
    })

interval_df = pd.DataFrame(interval_results)

display(interval_df.head(10))

,vm_id,duplicate_intervals,normal_intervals,short_intervals,long_intervals,large_gaps,max_gap_seconds
0,3,0,8628,0,1,4,1200.0
1,17,0,8630,0,2,3,900.0
2,23,0,8593,0,2,18,1200.0
3,28,0,8628,0,1,4,1200.0
4,36,0,8605,0,2,12,1500.0
5,40,0,8601,0,2,12,2100.0
6,55,0,8622,0,2,7,900.0
7,59,0,40,0,0,0,301.0
8,60,0,8628,0,2,4,900.0
9,63,0,8602,0,2,14,1200.0


In [13]:
print("VMs analyzed:", len(interval_df))

print("\n--- Interval Summary ---")

print(
    interval_df.drop(
        columns=["vm_id", "max_gap_seconds"]
    ).sum()
)

print(
    "\nMaximum observed gap (seconds):",
    interval_df["max_gap_seconds"].max()
)

print("\nVMs with the largest gaps:")

display(
    interval_df.sort_values(
        "max_gap_seconds",
        ascending=False
    ).head(10)
)

VMs analyzed: 100

--- Interval Summary ---
duplicate_intervals     32329
normal_intervals       787985
short_intervals         68705
long_intervals            194
large_gaps                993
dtype: int64

Maximum observed gap (seconds): 2100.0

VMs with the largest gaps:


,vm_id,duplicate_intervals,normal_intervals,short_intervals,long_intervals,large_gaps,max_gap_seconds
11,70,0,8596,0,2,15,2100.0
21,267,0,8604,0,2,10,2100.0
5,40,0,8601,0,2,12,2100.0
39,452,0,8597,0,2,12,2100.0
54,682,0,8599,0,2,13,2100.0
35,399,0,8594,0,2,16,1500.0
33,383,0,8593,0,3,15,1500.0
26,329,0,8596,0,2,15,1500.0
16,197,1338,7327,2608,2,10,1500.0
19,254,0,8595,0,2,15,1500.0


In [14]:
quality_results = []

numeric_columns = [
    "CPU usage [%]",
    "CPU usage [MHZ]",
    "CPU capacity provisioned [MHZ]",
    "Memory usage [KB]",
    "Memory capacity provisioned [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
    "Network received throughput [KB/s]",
    "Network transmitted throughput [KB/s]"
]

for file in selected_files:

    vm = pd.read_csv(file, sep=";")
    vm.columns = vm.columns.str.strip()

    measurements = vm[numeric_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    quality_results.append({
        "vm_id": file.stem,

        "invalid_numeric_cells": int(
            measurements.isna().sum().sum()
        ),

        "negative_measurements": int(
            (measurements < 0).sum().sum()
        ),

        "invalid_cpu_capacity": int(
            (measurements["CPU capacity provisioned [MHZ]"] <= 0).sum()
        ),

        "invalid_memory_capacity": int(
            (measurements["Memory capacity provisioned [KB]"] <= 0).sum()
        ),

        "cpu_pct_outside_range": int(
            (
                (measurements["CPU usage [%]"] < 0) |
                (measurements["CPU usage [%]"] > 100)
            ).sum()
        )
    })

quality_df = pd.DataFrame(quality_results)

display(quality_df.head(10))

,vm_id,invalid_numeric_cells,negative_measurements,invalid_cpu_capacity,invalid_memory_capacity,cpu_pct_outside_range
0,3,0,0,0,0,0
1,17,0,0,0,0,0
2,23,0,0,0,0,0
3,28,0,0,0,0,0
4,36,0,0,0,0,0
5,40,0,0,0,0,0
6,55,0,0,0,0,0
7,59,0,0,16,16,0
8,60,0,0,0,0,0
9,63,0,0,0,0,0


In [15]:
print("VMs validated:", len(quality_df))

print("\n--- Measurement Quality Summary ---")

print(
    quality_df.drop(columns="vm_id").sum()
)

print("\n--- VMs with potential issues ---")

display(
    quality_df[
        (quality_df.drop(columns="vm_id") > 0).any(axis=1)
    ]
)

VMs validated: 100

--- Measurement Quality Summary ---
invalid_numeric_cells           0
negative_measurements           0
invalid_cpu_capacity       105091
invalid_memory_capacity    105072
cpu_pct_outside_range       20126
dtype: int64

--- VMs with potential issues ---


,vm_id,invalid_numeric_cells,negative_measurements,invalid_cpu_capacity,invalid_memory_capacity,cpu_pct_outside_range
7,59,0,0,16,16,0
12,73,0,0,22581,22580,0
16,197,0,0,3999,3996,3919
17,220,0,0,4,4,1753
18,238,0,0,5,5,0
19,254,0,0,0,0,2618
20,264,0,0,0,0,2852
21,267,0,0,0,0,2705
22,273,0,0,0,0,1563
34,389,0,0,0,0,22
